In [0]:
# Bronze Layer: Raw Orders Ingestion
# Uses Auto Loader for incremental file ingestion
# Preserves raw data with added metadata columns

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    current_timestamp, lit,
    monotonically_increasing_id, col
)

spark = SparkSession.builder.getOrCreate()

# Configuration
LANDING_PATH  = "s3://ecom-landing-zone-2026/landing/orders/"
CHECKPOINT    = "s3://ecom-landing-zone-2026/checkpoints/orders_checkpoint/"
BRONZE_TABLE  = "ecomstore.ecomlake.bronze_orders"
SOURCE_SYSTEM = "order_management_system"

# Pre-create table with Auto-Compaction
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {BRONZE_TABLE} (
        order_id STRING, customer_id STRING, order_date STRING, order_time STRING, 
        status STRING, total_amount STRING, discount_amount STRING, final_amount STRING, 
        shipping_address_city STRING, shipping_address_state STRING, shipping_pincode STRING, 
        payment_method STRING, channel STRING, created_at STRING, updated_at STRING, 
        _source_system STRING, _batch_id STRING, _rescued_data STRING,
        _ingestion_timestamp TIMESTAMP, _source_file_name STRING, 
        _pipeline_layer STRING, _source_system_override STRING
    )
    USING DELTA
    TBLPROPERTIES (
        'delta.autoOptimize.optimizeWrite' = 'true',
        'delta.autoOptimize.autoCompact' = 'true'
    )
""")

# Schema Hint (optional but recommended for Auto Loader)
# We use PERMISSIVE mode — bad records are kept with nulls, not dropped
orders_schema_hints = """
    order_id STRING,
    customer_id STRING,
    order_date STRING,
    order_time STRING,
    status STRING,
    total_amount STRING,
    discount_amount STRING,
    final_amount STRING,
    shipping_address_city STRING,
    shipping_address_state STRING,
    shipping_pincode STRING,
    payment_method STRING,
    channel STRING,
    created_at STRING,
    updated_at STRING,
    _source_system STRING,
    _batch_id STRING
"""
# NOTE: All columns are STRING at bronze. We don't cast here.
# Bronze preserves raw data. Silver does the casting.

# Auto Loader Read
raw_orders_df = (
    spark.readStream
    .format("cloudFiles")                      # Auto Loader format
    .option("cloudFiles.format", "csv")        # source file format
    .option("cloudFiles.schemaLocation", CHECKPOINT + "schema/")
    .option("cloudFiles.inferColumnTypes", "false")   # keep everything as string
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")  # handle schema drift
    .option("cloudFiles.schemaHints", orders_schema_hints)
    .option("cloudFiles.rescuedDataColumn", "_rescued_data")
    .option("header", "true")
    .option("multiLine", "false")
    .option("mode", "PERMISSIVE")              # keep bad records
    .load(LANDING_PATH)
    # Add ingestion metadata columns 
    .withColumn("_ingestion_timestamp", current_timestamp())   # when did we ingest this?
    .withColumn("_source_file_name", col("_metadata.file_path"))        # which file did it come from?
    .withColumn("_pipeline_layer", lit("bronze"))
    .withColumn("_source_system_override", lit(SOURCE_SYSTEM))
)

# Write to Bronze Delta Table (Append Only)
(
    raw_orders_df.writeStream
    .format("delta")
    .outputMode("append")                      # bronze is ALWAYS append-only
    .option("checkpointLocation", CHECKPOINT)
    .option("mergeSchema", "true")             # allow schema evolution
    .trigger(availableNow=True)                # process all available files, then stop
    .toTable(BRONZE_TABLE)                     # writes to managed Delta table
)

print(f"✅ Bronze ingestion complete for: {BRONZE_TABLE}")